# Czech Bank Scoring
##### *Выполнил: Петрович Андрей, студент НИУ ВШЭ ОП "Экономика и анализ данных"*
##### В этом ноутбуке присутствует нормализация входных данных (processed data), создание модели машинного обучения и оценка ее работы

# Создание модели Логистической регрессии и нормализация данных

##### Шаг 1: Импорты библиотек

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.metrics import classification_report, recall_score, precision_score, confusion_matrix, roc_auc_score

##### Шаг 2: Чтение и объяснение данных 
Сделаем индексом первичный ключ таблицы `clean.loan`: `loan_id`.

In [2]:
df = pd.read_csv('../data/processed/features.csv')
df = df.set_index(df['loan_id']).drop('loan_id', axis = 1)
df.head()

,amount,duration,balance_mean_90d,balance_min_90d,n_trans_90d,n_negative_balance,n_sanction_interest,income_month_mean,payment_to_income,gender,age_at_loan,account_age_months,has_disponent,card_type,n_orders,district_avg_salary,bad_loan
loan_id,,,,,,,,,,,,,,,,,
4959,80952.0,24,30317.131579,16858.9,19,0,0,20672.363636,0.163165,M,48,10,1,none,1,12541,0
4961,30276.0,12,15123.228571,715.0,21,0,0,18411.153846,0.137036,F,57,12,0,none,0,9104,1
4962,30276.0,12,68836.628571,31621.9,28,0,0,62816.812500,0.040164,M,35,16,0,none,3,9893,0
4967,318480.0,60,20759.711111,-1011.2,27,1,1,40459.727273,0.131192,M,46,13,0,none,3,8547,1
4968,110736.0,48,26212.342308,15670.6,26,0,0,16365.555556,0.140967,F,58,8,0,none,3,8402,0


##### Рассмотрим колонки подробнее: 

In [3]:
df.info()

<class 'pandas.DataFrame'>
Index: 682 entries, 4959 to 7308
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   amount               682 non-null    float64
 1   duration             682 non-null    int64  
 2   balance_mean_90d     682 non-null    float64
 3   balance_min_90d      682 non-null    float64
 4   n_trans_90d          682 non-null    int64  
 5   n_negative_balance   682 non-null    int64  
 6   n_sanction_interest  682 non-null    int64  
 7   income_month_mean    682 non-null    float64
 8   payment_to_income    682 non-null    float64
 9   gender               682 non-null    str    
 10  age_at_loan          682 non-null    int64  
 11  account_age_months   682 non-null    int64  
 12  has_disponent        682 non-null    int64  
 13  card_type            682 non-null    str    
 14  n_orders             682 non-null    int64  
 15  district_avg_salary  682 non-null    int64  
 16  ba

##### 1. ***`amount`*** - Сумма кредита (чешские кроны).
##### 2. ***`duration`*** - Срок кредита в месяцах.
##### 3. ***`balance_mean_90d`*** - Средняя сумма остатка на счете за 90 дней до кредита (чешские кроны).
##### 4. ***`balance_min_90d`*** - Минимальный остаток счета за 90 дней до кредита (чешские кроны).
##### 5. ***`n_trans_90d`*** - Кол-во транзакций за 90 дней до кредита.
##### 6. ***`n_negative_balance`*** - Сколько раз за всю историю до кредита остаток уходил в минус.
##### 7. ***`n_sanction_interest`*** - Сколько раз банк начислял штрафной процент за отрицательный остаток.
##### 8. ***`income_month_mean`*** - Средний месячный приход на счёт (чешские кроны).
##### 9. ***`payment_to_income`*** - Отношение ежемесячного платежа к среднему месячному доходу.
##### 10. ***`gender`*** - Пол владельца счета (M - male, F - female).
##### 11. ***`age_at_loan`*** - Сколько лет было владельцу счета на дату получения кредита.
##### 12. ***`account_age_months`*** - Сколько месяцев счет был открыт до кредита.
##### 13. ***`has_disponent`*** - Есть ли у счета второй человек, который может им пользоваться (1 - да, 0 - нет).
##### 14. ***`card_type`*** - Тип карты (junior, classic, gold).
##### 15. ***`n_orders`*** - Число автоплатежей на счете.
##### 16. ***`district_avg_salary`*** - Средняя зарплата по району (чешские кроны).
##### 16. ***`n_orders`*** - Число автоплатежей на счете.
##### 17. ***`bad_loan`*** - Целевая переменная (1 - просрочили кредит, 0 - нет).

-----------------------------------------

##### Шаг 3: Dummy-переменные
Так, как логистическая регрессия работает только с численными переменными, то сделаем dummy переменные для строковых колонок `gender` и `card_type`.

In [4]:
df = pd.get_dummies(data = df, columns=['gender', 'card_type'], drop_first=True)
print(df.columns)

Index(['amount', 'duration', 'balance_mean_90d', 'balance_min_90d',
       'n_trans_90d', 'n_negative_balance', 'n_sanction_interest',
       'income_month_mean', 'payment_to_income', 'age_at_loan',
       'account_age_months', 'has_disponent', 'n_orders',
       'district_avg_salary', 'bad_loan', 'gender_M', 'card_type_gold',
       'card_type_junior', 'card_type_none'],
      dtype='str')


##### Шаг 4: Разделение на признаки (`X`) и целевую переменную (`y`)

In [5]:
X = df.drop(columns=['bad_loan'])
y = df['bad_loan']
print(y, '\n')
print(X)

loan_id
4959    0
4961    1
4962    0
4967    1
4968    0
       ..
7294    0
7295    0
7304    0
7305    0
7308    0
Name: bad_loan, Length: 682, dtype: int64 

           amount  duration  balance_mean_90d  balance_min_90d  n_trans_90d  \
loan_id                                                                       
4959      80952.0        24      30317.131579          16858.9           19   
4961      30276.0        12      15123.228571            715.0           21   
4962      30276.0        12      68836.628571          31621.9           28   
4967     318480.0        60      20759.711111          -1011.2           27   
4968     110736.0        48      26212.342308          15670.6           26   
...           ...       ...               ...              ...          ...   
7294      39168.0        24      59078.405000          41811.2           20   
7295     280440.0        60      46576.221053          20382.2           19   
7304     419880.0        60      63637.333333   

##### Шаг 5: Разбиение данных на обучающую / тестовую выборку и их анализ

In [6]:
X_train, X_test,y_train, y_test = train_test_split(X, y, test_size= 0.15, random_state=101, stratify=y)
print(f'Процент тестовой выборки совпадает с test_size = 0.15: {np.round((len(X_test) / len(X)), 2) == 0.15}')
pos_test = y_test.sum()
pos_train = y_train.sum()
print(f'Из {len(X_test)} наблюдений тестовой выборки ровно {pos_test} должников')
print(f'Из {len(X_train)} наблюдений обучающей выборки ровно {pos_train} должников')


Процент тестовой выборки совпадает с test_size = 0.15: True
Из 103 наблюдений тестовой выборки ровно 11 должников
Из 579 наблюдений обучающей выборки ровно 65 должников


##### Шаг 6: Нормализация данных
Используем для этого `StandartScaler`. Обучение (`fit`) проводим только на обучающей выборке во избежание утечек информации.

In [7]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print(X_train)


[[ 0.38858185  0.68112207  0.21447716 ... -0.05887448 -0.08340577
   0.2296242 ]
 [-0.66137246 -0.72986068 -1.29097054 ... -0.05887448 -0.08340577
   0.2296242 ]
 [ 1.73027181  0.68112207  1.0018519  ... -0.05887448 -0.08340577
  -4.3549417 ]
 ...
 [ 0.50199959 -0.72986068 -0.45321652 ... -0.05887448 -0.08340577
   0.2296242 ]
 [-0.95646864 -1.43535206  0.13027662 ... -0.05887448 -0.08340577
   0.2296242 ]
 [-0.71808134 -0.0243693   0.49045988 ... -0.05887448 -0.08340577
  -4.3549417 ]]


##### Шаг 7: Обучение модели логистической регрессии и нахождение оптимальных гиперпараметров
Для поиска оптимальных гиперпараметров будем использовать поиск по сетке (`GridSearchCV`)

In [8]:
param_grid = [
    {'penalty': ['l2'], 'solver': ['lbfgs', 'saga'],
     'C': [0.0001, 0.001, 0.01, 0.1, 1.0]},

    {'penalty': ['l1'], 'solver': ['saga'],
     'C': [0.0001, 0.001, 0.01, 0.1, 1.0]},

    {'penalty': ['elasticnet'], 'solver': ['saga'],
     'C': [0.0001, 0.001, 0.01, 0.1, 1.0, 10., 100.],
     'l1_ratio': [0.25, 0.5, 0.75]},
]
model = LogisticRegression(fit_intercept=True, max_iter=5000, class_weight='balanced')
gs = GridSearchCV(estimator= model, param_grid=param_grid, scoring='roc_auc', cv=5)
gs.fit(X_train, y_train)


/opt/miniconda3/envs/berka/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/berka/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/berka/lib/pyt

/opt/miniconda3/envs/berka/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/berka/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/berka/lib/pyt

/opt/miniconda3/envs/berka/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/berka/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/berka/lib/pyt

/opt/miniconda3/envs/berka/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/berka/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/opt/miniconda3/envs/berka/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio

/opt/miniconda3/envs/berka/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/berka/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/opt/miniconda3/envs/berka/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio

/opt/miniconda3/envs/berka/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/berka/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/berka/lib/pyt

... (16 outputs left out)

/opt/miniconda3/envs/berka/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/berka/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


/opt/miniconda3/envs/berka/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/berka/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


/opt/miniconda3/envs/berka/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/berka/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegre...max_iter=5000)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'C': [0.0001, 0.001, ...], 'penalty': ['l2'], 'solver': ['lbfgs', 'saga']}, {'C': [0.0001, 0.001, ...], 'penalty': ['l1'], 'solver': ['saga']}, ...]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support 

In [9]:
res = pd.Series(data = gs.best_params_, name='Значение параметра',)
res.head()

C           0.1
penalty      l1
solver     saga
Name: Значение параметра, dtype: object

#       Оценка модели и основные метрики ее работы

##### Шаг 8: Измерение качества модели на кросс-валидации (ROC-AUC)
Деление по умолчанию происходит на 5 фолдов (параметр `cv` = 5)

In [10]:
best_idx = gs.best_index_
cv_mean = gs.cv_results_['mean_test_score'][best_idx]
cv_std = gs.cv_results_['std_test_score'][best_idx]
folds = [gs.cv_results_[f'split{i}_test_score'][best_idx] for i in range(5)]
print(f'ROC-AUC на кросс-валидации: {np.round(cv_mean, 2)} +- {np.round(cv_std, 2)}')
print('Отдельно по фолдам: ')
for i in range(0, 5):
    print(f'фолд №{i+1}: ', np.round(folds[i], 4))

ROC-AUC на кросс-валидации: 0.91 +- 0.03
Отдельно по фолдам: 
фолд №1:  0.8559
фолд №2:  0.9193
фолд №3:  0.8984
фолд №4:  0.9305
фолд №5:  0.9502


**Вывод.** Средний ROC-AUC лучшей модели на 5-кратной кросс-валидации — **0.91 +- 0.03**.
Модель хорошо отделяет плохие кредиты от хороших: в 91% случаев случайно выбранному плохому кредиту она даёт
риск выше, чем случайно выбранному хорошему.
Разброс по фолдам — от 0.856 до 0.950.

##### Шаг 9: Предсказания модели


In [11]:
y_pred_proba = gs.predict_proba(X_test)
y_pred = gs.predict(X_test)
print("Предсказания (классы):\n",y_pred, '\n')
print("Предсказания (вероятности):\n",y_pred_proba[:5, 1])

Предсказания (классы):
 [0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0
 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 1 0 1 0 0 1 0 0 0 1 1 0 0 1 0 0 0 0 0
 0 0 0 1 1 0 0 0 1 0 1 0 0 0 1 0 0 0 1 0 0 0 0 1 0 0 0 0 0] 

Предсказания (вероятности):
 [0.20507096 0.03176877 0.00524304 0.43370689 0.60696609]


##### Шаг 10: ROC-AUC на тестовой выборке


In [12]:
print("ROC-AUC на тестовой выборке: ", np.round(roc_auc_score(y_true=y_test, y_score=y_pred_proba[:, 1]), 2))
print(f"ROC-AUC на кросс-валидации: {np.round(cv_mean, 2)} +- {np.round(cv_std, 2)}")

ROC-AUC на тестовой выборке:  0.83
ROC-AUC на кросс-валидации: 0.91 +- 0.03


**Вывод.** ROC-AUC на тестовой выборке — **0.83**, ниже, чем на кросс-валидации (0.91 +- 0.03).

Возможные причины:
- **маленький тест:** в нём 103 кредита и только 11 (`pos_test`) плохих — оценка AUC по 11 объектам очень шумная,
  один неверно упорядоченный плохой кредит заметно снижает её ;
- **оптимизм кросс-валидации:** `GridSearchCV` перебрал много комбинаций параметров и выбрал ту,
  что лучше всех сработала на этих же фолдах, поэтому `best_score_` немного завышен из-за максимума метрики по фолдам.

Итог: качество предложенной модели — **ROC-AUC ≈ 0.82–0.91**; это хороший уровень для кредитного скоринга.

##### Шаг 11: Анализ метрик precision, recall, classification report

In [13]:
print(f'precision: {np.round(precision_score(y_true=y_test, y_pred=y_pred), 3)}')
print(f'recall: {np.round(recall_score(y_true=y_test, y_pred=y_pred), 3)}')
print("\nClassification report:\n",classification_report(y_test, y_pred))

precision: 0.35
recall: 0.636

Classification report:
               precision    recall  f1-score   support

           0       0.95      0.86      0.90        92
           1       0.35      0.64      0.45        11

    accuracy                           0.83       103
   macro avg       0.65      0.75      0.68       103
weighted avg       0.89      0.83      0.85       103



**Вывод**:

Precision низкий, а recall средний - все это обусловлено `class_weight` = 'balanced'. Модель настроена не пропускать плохих, даже ценой лишних отказов. Для банка это обычно правильно, ведь пропущенный плохой кредит стоит дороже, чем отказ хорошему клиенту.

##### Шаг 12: Матрица ошибок


In [14]:
c = confusion_matrix(y_test, y_pred)
print(c)
print(c[0][0]," - правильная классификация кредитоспособного заемщика")
print(c[0][1], "- модель ошиблась и отказала кредитоспособному заемщику ")
print(c[1][0], "- модель ошиблась и выдала кредит некредитоспособному заемщику")
print(c[1][1], "- правильная классификация некредитоспособного заемщика")

[[79 13]
 [ 4  7]]
79  - правильная классификация кредитоспособного заемщика
13 - модель ошиблась и отказала кредитоспособному заемщику 
4 - модель ошиблась и выдала кредит некредитоспособному заемщику
7 - правильная классификация некредитоспособного заемщика


# Итог по модели

***Модель.*** Логистическая регрессия с L1-регуляризацией,
параметры подобраны через `GridSearchCV` по ROC-AUC на 5-кратной кросс-валидации.

***Качество.***
- ROC-AUC на кросс-валидации — **0.91 +- 0.03**, на  тестовой выборке — **0.83**;
- на тесте модель нашла **7 из 11** плохих кредитов (`recall` = 0.64), из 20 отказов обоснованы 7
  (`precision` = 0.35), что говорит о том, что модель настроена не пропускать плохие кредиты ценой лишних отказов.

**Ограничения.** Выборка маленькая (682 кредита, 76 плохих), поэтому метрики на тесте шумные;
кредиты 1998 года могли не успеть стать плохими.

##### Шаг 13: Сохраняем модель для дальнейшей визуализации

In [15]:
Path('../models').mkdir(exist_ok=True)
joblib.dump({
    "model": gs.best_estimator_,
    "X_test": X_test,
    "y_test": y_test,
    "feature_names": list(X.columns),
}, filename='../models/model.joblib')

['../models/model.joblib']